# Compare SAEs across sweep parameters (SAEBench)

This notebook loads **every SAE in one of your sweep runs** (e.g. `trained_models/Qwen3-8B/run10`)
and compares the members **against each other** — no pretrained-registry SAE involved. It works for a
**layers** sweep (members `L18, L22, …`), a **k** sweep (members `L24_k32, L24_k64, …`), or a single
run with multiple step checkpoints.

It reports exactly four metrics, all from [SAEBench](https://www.neuronpedia.org/sae-bench/info):

1. **Dead latents** — fraction of SAE features that never fire. For common widths you should get this
   well under 1%.
2. **Sparse probing** — SAEBench's sparse-probing benchmark (probe accuracy on top-k SAE latents
   across 35 binary tasks).
3. **L0 sparsity** — average number of active latents per token.
4. **Cross-entropy loss score** — how well the SAE reconstruction preserves the model's next-token loss
   (1.0 = perfect, 0.0 = zero-ablation).

Everything is driven by the **Parameters** cell: just point `RUN_DIR` at a run.

## Runtime notes

- Run this **inside the ROCm container** (the same env you train in). `sae-bench` is now in
  `requirements.txt`; install with `pip install -r requirements.txt` (it pulls `sae-probes`,
  `scikit-learn`, `xgboost`, and downgrades `pandas` to 2.x — no effect on training).
- **Cost:** the core eval and especially **sparse probing** run forward passes of the full base model
  and download several probing datasets, so a multi-member sweep on an 8B model takes a while. Sparse
  probing loads its **own** copy of the model, so this notebook frees the core-eval model first.
- **Dead latents** need enough tokens to be trustworthy — a latent looks "dead" if it simply never got
  a chance to fire. Raise `N_SPARSITY_BATCHES` if the dead count looks suspiciously high.

## Imports

In [ ]:
from IPython import get_ipython  # type: ignore

ipython = get_ipython()
if ipython is not None:
    ipython.run_line_magic("load_ext", "autoreload")
    ipython.run_line_magic("autoreload", "2")

import gc
import glob
import json
import re
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
import torch

from sae_lens import SAE
from sae_lens.training.activations_store import ActivationsStore
from transformer_lens import HookedTransformer

# SAEBench: core metrics (dead latents, L0, CE-loss-score) + sparse probing.
from sae_bench.evals.core.main import run_evals as core_run_evals
from sae_bench.evals.core.eval_config import CoreEvalConfig
from sae_bench.evals.sparse_probing.main import run_eval as sparse_probing_run_eval
from sae_bench.evals.sparse_probing.eval_config import SparseProbingEvalConfig
from sae_bench.sae_bench_utils import general_utils

torch.set_grad_enabled(False)
pd.set_option("display.max_rows", 200)
print("torch:", torch.__version__, "| cuda available:", torch.cuda.is_available())

## 1. Parameters

Edit these to point at any run you've trained and any matched pretrained SAE.

In [ ]:
# === The sweep to compare ====================================================
# Point at ONE run directory from train_interactive.sh. Every SAE inside is
# discovered and compared against the others (layers sweep, k sweep, or a single
# run's step checkpoints).
RUN_DIR = "/wekafs/smerrill/efficient_sae/trained_models/Qwen3-8B/run3"

# === Eval dataset for the core metrics (config-free so it streams cleanly) ===
EVAL_DATASET = "Skylion007/openwebtext"
CONTEXT_SIZE = 128
EVAL_BATCH_SIZE_PROMPTS = 16

# --- Core eval: dead latents / L0 / CE-loss-score ----------------------------
N_RECON_BATCHES = 20       # batches for CE-loss-score
N_SPARSITY_BATCHES = 500   # batches for L0 + feature density (dead latents)
DEAD_LATENT_THRESHOLD = 0.0  # a latent with density <= this (never fired) is "dead"

# --- Sparse probing ----------------------------------------------------------
SP_K_VALUES = [1, 2, 5]    # number of top SAE latents the probe trains on
SP_LLM_BATCH_SIZE = 16
SP_OUTPUT_DIR = Path(RUN_DIR) / "eval_results" / "sparse_probing"

# === Hardware ================================================================
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
# Keep the model + SAEs in the SAME dtype to avoid hook dtype mismatches.
# bf16 keeps the 8B model ~16GB; use "float32" for max precision.
DTYPE = "bfloat16"

print(f"run_dir = {RUN_DIR}")
print(f"device  = {DEVICE} | dtype = {DTYPE}")
print(f"eval dataset = {EVAL_DATASET} (ctx {CONTEXT_SIZE})")

## 2. Discover the sweep-member SAEs

We scan `RUN_DIR` for every `sae_weights.safetensors`, group them by sweep member, and keep the **final**
checkpoint (largest step) of each. Each member is labelled from its own `cfg.json` (hook layer, and `k`
for TopK), so labels are meaningful for both layouts:

- sweep: `<run>/checkpoints/<id>/<ckpt>/<key>/`
- single: `<run>/L<layer>/<wandb_id>/<step>/`

In [ ]:
run_dir = Path(RUN_DIR)
run_cfg_path = run_dir / "config.json"
run_config = json.loads(run_cfg_path.read_text()) if run_cfg_path.exists() else {}


def _step_of(checkpoint_dir: Path) -> int:
    """Training step encoded in a checkpoint dir name ('final_123' or '123')."""
    m = re.search(r"\d+", checkpoint_dir.name)
    return int(m.group(0)) if m else -1


def _member_label(weights_path: Path) -> str:
    """A stable, meaningful label for the sweep member a checkpoint belongs to,
    read from the SAE's own cfg.json (hook layer + k) rather than the dir name."""
    cfg = json.loads((weights_path.parent / "cfg.json").read_text())
    hook = cfg.get("metadata", {}).get("hook_name", "")
    lm = re.search(r"blocks\.(\d+)\.", hook)
    layer = lm.group(1) if lm else "?"
    k = cfg.get("k")
    return f"L{layer}" + (f"_k{k}" if k is not None else "")


def discover_saes(run_dir: Path) -> dict[str, Path]:
    """Map sweep-member label -> final checkpoint dir (largest step) for that member."""
    weights = [Path(p) for p in glob.glob(str(run_dir / "**" / "sae_weights.safetensors"), recursive=True)]
    if not weights:
        raise FileNotFoundError(f"No sae_weights.safetensors found under {run_dir}")
    best: dict[str, Path] = {}
    best_step: dict[str, int] = {}
    for w in weights:
        label, step = _member_label(w), _step_of(w.parent)
        if label not in best or step > best_step[label]:
            best[label], best_step[label] = w.parent, step
    return dict(sorted(best.items()))


sae_dirs = discover_saes(run_dir)

print(f"Run: {run_dir.parent.name}/{run_dir.name}  ({len(sae_dirs)} SAEs to compare)\n")
if run_config:
    print("run-level config.json:")
    print(json.dumps(run_config, indent=2))
print("\nDiscovered SAEs (final checkpoint per member):")
for label, d in sae_dirs.items():
    print(f"  {label:12s} <- {d}")

## 3. Load the SAEs

Each member loads from disk onto the same device/dtype. We then standardize its config so SAEBench can
read `cfg.hook_name` / `cfg.hook_layer` (SAELens v6 keeps these on `cfg.metadata`).

In [ ]:
saes: dict[str, SAE] = {}
for label, d in sae_dirs.items():
    sae = SAE.load_from_disk(str(d), device=DEVICE, dtype=DTYPE)
    # Standardize cfg in-place so SAEBench can read cfg.hook_name / cfg.hook_layer.
    general_utils.load_and_format_sae(label, sae, DEVICE)
    saes[label] = sae

print(f"Loaded {len(saes)} SAEs:\n")
for label, sae in saes.items():
    md = sae.cfg.metadata
    arch = sae.cfg.architecture() if callable(sae.cfg.architecture) else sae.cfg.architecture
    print(f"  {label:12s} {type(sae).__name__:20s} {str(arch):10s} "
          f"d_in={sae.cfg.d_in} d_sae={sae.cfg.d_sae} (x{sae.cfg.d_sae // sae.cfg.d_in})  "
          f"hook={md.hook_name}")

# All members share one base model.
MODEL_NAME = next(iter(saes.values())).cfg.metadata.model_name
print(f"\nbase model = {MODEL_NAME}")

## 4. Load the base model

All members target the same base model, so we load it **once** for the core eval (each member reads
activations at its own hook). Loading an 8B model takes a minute. (Sparse probing later loads its own
copy, so we free this one before running it.)

In [ ]:
model_kwargs = dict(next(iter(saes.values())).cfg.metadata.model_from_pretrained_kwargs or {})
print(f"Loading {MODEL_NAME} (dtype={DTYPE}) ... this can take a minute.")
model = HookedTransformer.from_pretrained_no_processing(
    MODEL_NAME, device=DEVICE, dtype=DTYPE, **model_kwargs,
)
print("Model loaded:", model.cfg.model_name, "| n_layers:", model.cfg.n_layers)

## 5. Core metrics — dead latents, L0, CE-loss-score (SAEBench `core`)

For each member we build an `ActivationsStore` at its hook and run SAEBench's `core` eval. From it we
take:
- **L0** — `sparsity.l0`
- **CE-loss-score** — `model_performance_preservation.ce_loss_score`
- **Dead latents** — features whose `feature_density` is `<= DEAD_LATENT_THRESHOLD` (never fired)

In [ ]:
core_config = CoreEvalConfig(
    batch_size_prompts=EVAL_BATCH_SIZE_PROMPTS,
    n_eval_reconstruction_batches=N_RECON_BATCHES,
    n_eval_sparsity_variance_batches=N_SPARSITY_BATCHES,
    compute_ce_loss=True,
    compute_sparsity_metrics=True,
    compute_featurewise_density_statistics=True,
    exclude_special_tokens_from_reconstruction=True,
    dataset=EVAL_DATASET,
    context_size=CONTEXT_SIZE,
    llm_dtype=DTYPE,
    verbose=False,
)

core_results: dict[str, dict] = {}
for label, sae in saes.items():
    print(f"\n=== core eval: {label}  (hook={sae.cfg.hook_name}) ===")
    store = ActivationsStore.from_sae(
        model, sae,
        dataset=EVAL_DATASET,
        context_size=CONTEXT_SIZE,
        store_batch_size_prompts=EVAL_BATCH_SIZE_PROMPTS,
        device=DEVICE,
    )
    all_metrics, feature_metrics = core_run_evals(sae, store, model, core_config, verbose=False)

    fd = np.asarray(feature_metrics.get("feature_density", []), dtype=np.float64)
    n_dead = int((fd <= DEAD_LATENT_THRESHOLD).sum()) if fd.size else 0
    core_results[label] = {
        "l0": float(all_metrics["sparsity"]["l0"]),
        "ce_loss_score": float(all_metrics["model_performance_preservation"]["ce_loss_score"]),
        "n_features": int(fd.size),
        "n_dead": n_dead,
        "dead_pct": (100.0 * n_dead / fd.size) if fd.size else float("nan"),
        "feature_density": fd,
    }
    r = core_results[label]
    print(f"  L0={r['l0']:.2f}  CE-loss-score={r['ce_loss_score']:.4f}  "
          f"dead={r['n_dead']}/{r['n_features']} ({r['dead_pct']:.3f}%)")
    del store
    gc.collect()
    torch.cuda.empty_cache()

## 6. Sparse probing (SAEBench)

SAEBench's sparse-probing benchmark encodes inputs through each SAE, mean-pools over tokens, selects the
top-`k` latents by mean difference, trains a logistic-regression probe, and reports test accuracy across
35 binary tasks (bias_in_bios, amazon reviews, github, ag_news, europarl). Higher is better; the LLM
residual-stream probe accuracy is reported alongside as a baseline.

`run_eval` loads its **own** copy of the base model, so we free the core-eval model first. This step
downloads the probing datasets and is the slowest part of the notebook.

In [ ]:
# Sparse probing loads its OWN copy of the base model, so free ours first.
try:
    del model
except NameError:
    pass
gc.collect()
torch.cuda.empty_cache()

sp_config = SparseProbingEvalConfig(
    model_name=MODEL_NAME,
    llm_dtype=DTYPE,
    llm_batch_size=SP_LLM_BATCH_SIZE,
    k_values=SP_K_VALUES,
)
selected_saes = [(label, sae) for label, sae in saes.items()]

SP_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
sp_raw = sparse_probing_run_eval(
    sp_config,
    selected_saes,
    device=DEVICE,
    output_path=str(SP_OUTPUT_DIR),
    force_rerun=True,
    clean_up_activations=True,
    save_activations=False,
)

# Headline sparse-probing accuracy (top-k SAE latents) + the LLM baseline.
sparse_probing_results: dict[str, dict] = {}
for label in saes:
    metrics = sp_raw[f"{label}_custom_sae"]["eval_result_metrics"]
    sae_m, llm_m = metrics["sae"], metrics["llm"]
    sparse_probing_results[label] = {
        **{f"sae_top_{k}_acc": sae_m.get(f"sae_top_{k}_test_accuracy") for k in SP_K_VALUES},
        **{f"llm_top_{k}_acc": llm_m.get(f"llm_top_{k}_test_accuracy") for k in SP_K_VALUES},
    }
    r = sparse_probing_results[label]
    print(f"  {label:12s} " + "  ".join(
        f"top{k}: SAE={r[f'sae_top_{k}_acc']:.3f} (LLM {r[f'llm_top_{k}_acc']:.3f})"
        for k in SP_K_VALUES))

## 7. Results table

The four metrics, one row per sweep member. `dead_latents_%` lower is better; `CE_loss_score` and the
sparse-probing accuracy higher is better; `L0` is the achieved sparsity (interpret relative to your
target, not as "higher/lower better").

In [ ]:
SP_MAIN_K = SP_K_VALUES[0]  # headline sparse-probing k (top-1 by default)

rows = []
for label in saes:
    c = core_results.get(label, {})
    s = sparse_probing_results.get(label, {})
    rows.append({
        "SAE": label,
        "dead_latents_%": c.get("dead_pct"),
        "n_dead": c.get("n_dead"),
        "L0": c.get("l0"),
        "CE_loss_score": c.get("ce_loss_score"),
        f"sparse_probe_top{SP_MAIN_K}_acc": s.get(f"sae_top_{SP_MAIN_K}_acc"),
        f"llm_top{SP_MAIN_K}_acc": s.get(f"llm_top_{SP_MAIN_K}_acc"),
    })

summary_df = pd.DataFrame(rows).set_index("SAE")
display(summary_df)

In [ ]:
# One bar chart per metric, across sweep members.
bar_specs = [
    ("dead_latents_%", "Dead latents (%) — lower is better"),
    ("L0", "L0 sparsity"),
    ("CE_loss_score", "CE loss score — higher is better"),
    (f"sparse_probe_top{SP_MAIN_K}_acc", f"Sparse probing top-{SP_MAIN_K} accuracy — higher is better"),
]
for col, title in bar_specs:
    if col not in summary_df.columns:
        continue
    sub = summary_df.reset_index()[["SAE", col]].dropna()
    if sub.empty:
        continue
    fig = px.bar(sub, x="SAE", y=col, title=title, text_auto=".3f")
    fig.update_layout(height=380, width=720, xaxis_title="", yaxis_title="")
    fig.show()

## 8. Feature-density histograms (the visual companion to the dead-latent count)

Overlaid **log10 feature-density** histograms across sweep members. A spike at the far left (~ -10) is
the pile of dead features; a bump near 0/-1 is dense (often less interpretable) features.

In [ ]:
density_frames = []
for label, c in core_results.items():
    fd = c.get("feature_density")
    if fd is None or len(fd) == 0:
        print(f"(no feature_density for {label})")
        continue
    df = pd.DataFrame({"log10_feature_density": np.log10(np.asarray(fd) + 1e-10)})
    df["SAE"] = label
    density_frames.append(df)

if density_frames:
    dens = pd.concat(density_frames, ignore_index=True)
    fig = px.histogram(
        dens, x="log10_feature_density", color="SAE", nbins=100, barmode="overlay",
        opacity=0.6, title="Log10 feature density (spike at ~ -10 = dead latents)",
    )
    fig.update_layout(height=450, width=900, legend_title="")
    fig.show()
else:
    print("No feature-density metrics available to plot.")

## Notes & next steps

- **Reading the comparison:**
  - *Dead latents* — aim for well under 1%. A high count on every member usually means too few eval
    tokens; raise `N_SPARSITY_BATCHES` before concluding the SAE is bad.
  - *CE-loss-score* — closer to 1.0 is better (the SAE reconstruction barely hurts the model).
  - *Sparse probing* — higher SAE accuracy is better; compare against the LLM-baseline column to see how
    much of the linear-probe signal the SAE preserves.
  - *L0* — the achieved sparsity. For a k-sweep this should track your target `k`; use it to read the
    other three metrics along the sparsity axis.
- **Cross-member trends:** for a layers sweep, watch how dead-latent % and sparse-probing accuracy move
  with depth; for a k sweep, watch the sparsity↔fidelity tradeoff (higher k → better CE-loss-score but
  often worse interpretability).
- **Results are cached** under `<run>/eval_results/sparse_probing/` (one JSON per member). Set
  `force_rerun=False` in section 6 to reuse them.
- **More members:** anything `discover_saes` finds is compared automatically — point `RUN_DIR` at a
  different run, or drop more checkpoints into the run directory.